<a href="https://colab.research.google.com/github/trainocate-japan/developing-agentic-ai-with-langchain/blob/main/chap04/exercise/solution/chap04_exercise_4B_solution.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 演習 4-B【正解 (solution)】: 会話を記憶するヘルプデスク — ヘルプデスク Step 3

**研修コース「Agentic AI 開発実践 - LangChain 版」/ 第4章「メモリと可観測性」**

この Notebook は演習 4-B の**正解 (solution)** です。
`# TODO` を埋める前に答えを見てしまわないよう、**まずは starter で自力で挑戦**してください。
詰まったとき・答え合わせのときにこちらを参照しましょう。

## この演習で作るもの

第3章 (Step 2) で作った「**ヘルプデスクエージェント v1**」(FAQ 検索 + 稼働状況ツール + 構造化出力) は、
問い合わせのたびに会話を忘れてしまいます。これを本章で学んだ部品で拡張し、

- **社員ごとに会話を記憶する** (checkpointer + 社員 ID を `thread_id` に対応)
- **運用チームがトレースで診断できる** (LangSmith に `tags` / `metadata` 付きで記録)

という「**ヘルプデスクエージェント v2**」に育てます。

完成すると、「私は人事部の佐藤です。VPN に繋がりません」→「さっき伝えた所属はどこ?」のような
**2 ターンの会話を記憶**し、別の社員 ID では記憶が分離され、利用部署をタグでトレースに残せるエージェントが
手に入ります。

## ヘルプデスク演習ストーリーにおける位置づけ (Step 3)

本コースの演習は「**社内 IT ヘルプデスクエージェント**」を第2〜8章で段階的に拡張して完成させます。
この演習はその **Step 3** にあたります。

| 章 | 追加する要素 | 演習後の姿 |
|---|---|---|
| 第2章 | Function Calling 手動ループ | 稼働状況に答える素朴な QA ループ (openai 直接) |
| 第3章 | create_agent / @tool / 構造化出力 | FAQ 検索 + 稼働状況ツールを持つ単体エージェント (v1) |
| **第4章 (この演習)** | **Checkpointer / LangSmith** | **社員ごとに会話を記憶し、トレースで診断できるエージェント (v2)** |
| 第5章以降 | MCP / HITL / 評価 / マルチエージェント | … 最終的に Web UI から操作できるヘルプデスクへ |

> **第3章の成果物は不要です。** v1 のツール (`search_faq` / `get_system_status`)・FAQ データ・
> `SupportAnswer` はこの Notebook に**同梱済み**なので、第3章の演習が未完了でも、この Notebook だけで完結します。

## 前提条件

- このファイルを **Google Colab** で開いていること
- Colab の **[シークレット]** に **2 種類のキー**が登録されていること
  - **`OPENAI_API_KEY`** — 第1章の演習 1-1 で登録済みのはずです (未登録でも「0. セットアップ」で登録できます)
  - **`LANGSMITH_API_KEY`** — 本章で**新しく必要**になります。
    「0. セットアップ」の手順で、無料アカウント作成 → API キー発行 → シークレット登録を行います
- インターネット接続 (API を呼び出します)

## 所要時間

約 15 分

---
> **モデル名について**: モデル名は変数 `MODEL` に集約しています (例: `MODEL = "openai:gpt-5.4"`)。
> 研修実施時は講師が指定する最新モデル名に差し替えてください。

## 0. セットアップ

### 0-1. 依存パッケージのインストール

LangChain v1 本体 (`langchain`) と OpenAI 統合 (`langchain-openai`) をインストールします。
checkpointer (`InMemorySaver`) は `langgraph` 同梱、LangSmith トレースも追加パッケージ不要です。

> 研修実施時は再現性のため、バージョンをピン留めすることを推奨します
> (本コースの基盤は **langchain 1.3.x / langchain-openai 1.3.x** です)。

In [ ]:
# LangChain v1 本体と OpenAI 統合を最新版へインストール/更新
# 研修実施時はバージョンをピン留め推奨 (langchain 1.3.x / langchain-openai 1.3.x)
!pip install -U langchain langchain-openai

### 0-2. OpenAI API キーのセットアップ (Colab シークレット方式)

API キーは**コードに直接書かず**、Colab の **[シークレット]** 機能で管理します。

**操作手順** (未登録の場合):
1. 画面左の **鍵アイコン 🔑 [シークレット]** をクリック
2. **[新しいシークレットを追加]** を押す
3. 名前に `OPENAI_API_KEY`、値にあなたの API キーを入力
4. このノートブックからのアクセスを **オン** にする

In [ ]:
import os

# Colab のシークレットから OpenAI API キーを読み込み、環境変数に設定する
# Colab 以外の環境では except 側に入り、既存の環境変数 OPENAI_API_KEY をそのまま使う
try:
    from google.colab import userdata
    os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")
    print("Colab シークレットから OPENAI_API_KEY を読み込みました。")
except ImportError:
    print("Colab 以外の環境です。環境変数 OPENAI_API_KEY を使用します。")

# モデル名は変数に集約 ("provider:model" 形式)。研修実施時に最新へ差し替え
MODEL = "openai:gpt-5.4"

print("OpenAI APIキー設定済み:", bool(os.environ.get("OPENAI_API_KEY")))
print("使用モデル:", MODEL)

### 0-3. LangSmith のセットアップ (本章で新規に必要)

この演習の後半では **LangSmith** でトレースを読み解きます。LangSmith の無料アカウントと API キーが必要です。
**まだ用意していない場合は、次の手順で準備してください** (所要 2〜3 分、クレジットカード不要)。

**操作手順**:
1. ブラウザで [smith.langchain.com](https://smith.langchain.com) を開き、**無料アカウントを作成**してログインする
2. 左下の **Settings** → **API Keys** を開く
3. **[Create API Key]** を押して API キーを発行し、表示された文字列を**コピー**する (発行時のみ表示)
4. Colab に戻り、左サイドバーの **🔑 [シークレット]** で **[新しいシークレットを追加]**
5. 名前に **`LANGSMITH_API_KEY`**、値にコピーしたキーを貼り付け、**アクセスをオン**にする

> **トレースの有効化に必要なのは「環境変数 2 つだけ」**です (`LANGSMITH_TRACING` と `LANGSMITH_API_KEY`)。
> エージェントのコードには一切手を入れません。次のセルでトレースを有効化します。

In [ ]:
# --- LangSmith トレースの有効化 (コード変更ゼロ。環境変数を設定するだけ) ---
os.environ["LANGSMITH_TRACING"] = "true"                      # トレース送信を有効化
os.environ["LANGSMITH_PROJECT"] = "langchain-training-day1"   # 送信先プロジェクト名 (整理用)

# Colab シークレットから LangSmith API キーを読み込む
# 非 Colab では環境変数 LANGSMITH_API_KEY を設定済みとみなす
try:
    from google.colab import userdata
    os.environ["LANGSMITH_API_KEY"] = userdata.get("LANGSMITH_API_KEY")
except Exception:
    pass

print("LANGSMITH_TRACING :", os.environ.get("LANGSMITH_TRACING"))
print("LANGSMITH_PROJECT :", os.environ.get("LANGSMITH_PROJECT"))
print("LANGSMITH_API_KEY 設定済み:", bool(os.environ.get("LANGSMITH_API_KEY")))

---

## 1. ヘルプデスクエージェント v1 を同梱する (第3章の成果物・配布済み)

ここは**配布済み**のコードです (実行するだけ)。第3章で作った v1 の構成部品——
**稼働状況ツール `get_system_status`**、**FAQ データ `FAQ_DATA`**、**FAQ 検索ツール `search_faq`**、
**構造化出力スキーマ `SupportAnswer`**——をまとめて用意します。

第3章の演習をやっていなくても、ここから始められます。

In [ ]:
from langchain.tools import tool
from pydantic import BaseModel, Field

# --- 配布: 社内システムの稼働状況を返すダミーデータ ---
SYSTEM_STATUS = {
    "勤怠システム": "正常稼働中",
    "経費精算システム": "正常稼働中",
    "メールサーバー": "一部遅延あり (調査中)",
    "VPN": "メンテナンス中 (本日 22:00 まで)",
}


@tool
def get_system_status(service: str) -> str:
    """指定された社内システムの現在の稼働状況を取得する。

    システムが「動いているか」「障害が出ていないか」「メンテナンス中か」といった
    稼働状態の問い合わせに使う。

    Args:
        service: 稼働状況を知りたい社内システムの名称 (例: 勤怠システム, VPN)
    """
    status = SYSTEM_STATUS.get(service, "不明 (登録されていないシステムです)")
    return f"{service}の稼働状況: {status}"


# --- 配布: FAQ 検索ツールが参照する FAQ データ ---
FAQ_DATA = {
    "VPN": "VPN に接続できない場合は、まず社内ポータルから最新の VPN クライアントを再インストールし、"
           "二要素認証アプリの時刻同期を確認してください。それでも解決しない場合は情報システム部へ。",
    "パスワード": "パスワードを忘れた場合は、社内ポータルの『パスワード再設定』から手続きできます。"
                  "ロックされた場合の解除は、本人確認のうえ情報システム部での対応が必要です。",
    "経費精算": "経費精算は経費精算システムから申請します。領収書は PDF で添付し、月末締め翌月 10 日払いです。",
    "メール": "メールの容量超過時は、添付ファイルの大きい古いメールを削除するか、アーカイブを利用してください。",
}


@tool
def search_faq(keyword: str) -> str:
    """社内 FAQ をキーワードで検索し、よくある質問への回答を返す。

    VPN・パスワード・経費精算・メールなど、社内の手続きややり方に関する
    「どうすればよいか」という問い合わせに使う。稼働状況の確認には使わない。

    Args:
        keyword: 検索キーワード (例: VPN, パスワード, 経費精算)
    """
    hits = [text for key, text in FAQ_DATA.items() if keyword in key or key in keyword]
    if hits:
        return "\n".join(hits)
    return "該当する FAQ が見つかりませんでした。"


# --- 配布: 構造化出力スキーマ (第3章で作成) ---
class SupportAnswer(BaseModel):
    """ヘルプデスクの一次対応の回答。チケット管理システムに渡せる構造化データ。"""

    category: str = Field(
        description="問い合わせの分類 (例: VPN, パスワード, 経費精算, 稼働状況)"
    )
    answer: str = Field(
        description="ユーザーに提示する回答本文。簡潔で分かりやすい一次対応の案内。"
    )
    escalation_required: bool = Field(
        description="情報システム部への引き継ぎ (エスカレーション) が必要なら True。"
                    "本人確認やアカウントのロック解除など、一次対応で完結しない場合に True にする。"
    )


# 動作確認 (配布部分が読み込めたか)
print("FAQ キーワード:", list(FAQ_DATA.keys()))
print(get_system_status.invoke({"service": "勤怠システム"}))
print("SupportAnswer フィールド:", list(SupportAnswer.model_fields.keys()))

---

## 2. エージェントに checkpointer を追加する【TODO①】

v1 を「会話を記憶する v2」にする最初のステップです。`create_agent` に **checkpointer** を渡します。
開発・検証用の checkpointer として `InMemorySaver` (プロセス内メモリに状態を保存) を使います。

**TODO① でやること**:
- `InMemorySaver` を **import** する
- `create_agent` に **`checkpointer=InMemorySaver()`** を渡す

> **ヒント**: checkpointer を渡すと、エージェントはステップごとに state (会話履歴) を保存するようになります。
> **会話の記憶にはこれが必要**です。import 元は `from langgraph.checkpoint.memory import InMemorySaver`、
> `create_agent` の `model` / `tools` / `system_prompt` / `response_format` に**もう 1 つ引数を足す**だけです。

In [ ]:
from langchain.agents import create_agent
from langgraph.checkpoint.memory import InMemorySaver   # [TODO①] checkpointer を import

agent = create_agent(
    model=MODEL,
    tools=[search_faq, get_system_status],
    system_prompt=(
        "あなたは社内 IT ヘルプデスクの一次対応担当です。"
        "社員からの問い合わせに、丁寧かつ簡潔に答えてください。"
        "手続きややり方の質問には FAQ 検索ツール (search_faq) を、"
        "システムが動いているかなどの稼働状況の質問にはステータス確認ツール (get_system_status) を使って調べます。"
        "本人確認やアカウント操作など一次対応で完結しない場合は、情報システム部へのエスカレーションを案内してください。"
    ),
    response_format=SupportAnswer,
    checkpointer=InMemorySaver(),   # [TODO①] state の保存係を渡す → これで会話を記憶できる
)

print("ヘルプデスクエージェント v2 (記憶あり) を構成しました。")

---

## 3. 社員 ID をスレッドにして会話を記憶する【TODO②】

`thread_id` は「**会話の鍵**」です。これに**社員 ID** を対応させると、「社員ごとに別々の会話を記憶する
エージェント」が自然に実現できます。同じ社員 ID (= 同じ鍵) で invoke すれば会話が継続し、
別の社員 ID なら記憶は分離されます。

ここでは社員「佐藤」さん (社員 ID `emp-sato`) の 2 ターンの会話を試します。

1. ターン 1: 「私は人事部の佐藤です。VPN に繋がりません。」
2. ターン 2: 「さっき伝えた所属はどこ?」 ← **1 ターン目を覚えていれば「人事部」と答えられる**

**TODO② でやること**:
- `config` を **`{"configurable": {"thread_id": <社員ID>}}`** の形で構成する (社員 ID をスレッドに)

> **ヒント**: `thread_id` は「会話の鍵」。**同じ鍵なら続き、違う鍵なら新規**です。
> ここでは社員 ID 文字列 (`"emp-sato"`) を `thread_id` に渡します。

In [ ]:
# [TODO②] 社員 ID をスレッドにする (社員 ID = 会話の鍵)
EMP_SATO = "emp-sato"
config_sato = {"configurable": {"thread_id": EMP_SATO}}

# ターン 1: 所属と困りごとを伝える
res1 = agent.invoke(
    {"messages": [{"role": "user", "content": "私は人事部の佐藤です。VPN に繋がりません。"}]},
    config_sato,
)
print("【ターン1】", res1["structured_response"].answer)
print()

# ターン 2: 同じ thread_id (= 同じ社員) で、前のターンの内容を尋ねる
res2 = agent.invoke(
    {"messages": [{"role": "user", "content": "さっき伝えた所属はどこ?"}]},
    config_sato,
)
print("【ターン2】", res2["structured_response"].answer)   # => 「人事部」と答えられれば記憶できている
print("messages 件数 (累積):", len(res2["messages"]))

### 別の社員 ID では記憶がないことを確認する

次に、別の社員「田中」さん (社員 ID `emp-tanaka`) で「私の所属はどこ?」と聞いてみます。
田中さんのスレッドには佐藤さんの会話は存在しないので、エージェントは所属を知らないはずです。

**期待される結果**: 別の社員 ID は別の会話なので、所属を答えられません
(「伺っていません」といった応答になります)。これが「社員ごとに記憶が分離される」ことの確認です。

In [ ]:
# 別の社員 ID = 別のスレッド = 別の会話 (佐藤さんの記憶は引き継がれない)
config_tanaka = {"configurable": {"thread_id": "emp-tanaka"}}

res_other = agent.invoke(
    {"messages": [{"role": "user", "content": "私の所属はどこ?"}]},
    config_tanaka,
)
print("【別の社員 (田中)】", res_other["structured_response"].answer)   # => 所属を知らない

---

## 4. トレースに利用部署を記録する【TODO③】

運用チームが「どの部署からの問い合わせか」をトレースで追えるように、`config` に **`tags`** と
**`metadata`** を付けます。`tags` は検索用のラベル、`metadata` は任意のキーバリューです。

**重要**: `tags` と `metadata` は `configurable` の**中ではなく、同じ階層**に書きます。

```python
config = {
    "configurable": {"thread_id": "emp-sato"},   # 会話の鍵
    "tags": [...],                               # ← configurable と同じ階層
    "metadata": {...},                           # ← configurable と同じ階層
}
```

**TODO③ でやること**:
- 佐藤さん用の `config` に **`tags`** (例: `["day1", "helpdesk"]`) と
  **`metadata`** (例: `{"department": "人事部"}`) を追加する

> **ヒント**: **`tags` は `configurable` と同じ階層**に書きます (`configurable` の中ではありません)。
> このセルは佐藤さんのスレッド (`emp-sato`) の続きなので、トレースには 2 ターン分の会話が記録されます。

In [ ]:
# [TODO③] tags / metadata を configurable と同じ階層に追加する
config_sato_tagged = {
    "configurable": {"thread_id": EMP_SATO},   # 会話の鍵 (佐藤さんの続き)
    "tags": ["day1", "helpdesk"],              # 検索用ラベル (configurable と同じ階層)
    "metadata": {"department": "人事部"},       # 利用部署を記録 (configurable と同じ階層)
}

# 佐藤さんのスレッドで、もう 1 ターン (タグ付きでトレースに記録される)
res3 = agent.invoke(
    {"messages": [{"role": "user", "content": "VPN がダメなとき、ほかに試せることはありますか?"}]},
    config_sato_tagged,
)
print("【ターン3 (タグ付き)】", res3["structured_response"].answer)
print()
print("→ smith.langchain.com の 'langchain-training-day1' で、tags='helpdesk' のトレースを探してください。")
print("   metadata の department='人事部' も記録されています。")

---

## 5. トレース読解ワークシート — 2 ターン目を読み解く

最後に、LangSmith のトレースから値を読み取ります。ブラウザで
[smith.langchain.com](https://smith.langchain.com) を開き、プロジェクト `langchain-training-day1` から
**セクション 3 のターン 2** (「さっき伝えた所属はどこ?」) のトレースを開いてください。

### 3 点チェックで読む

1. **① ループは何周したか** — `model → tools` の繰り返し回数を数えます。
2. **② どのツールが、どの引数で呼ばれたか** — 各 run をクリックして入出力を見ます。
3. **③ トークンをどこで消費したか** — run ごとの入出力トークン数を見ます。

### ワークシート (記入例つき)

> **注**: 下の「記入例」はあくまで一例です。**実際の値は実行・モデルにより変わります**。
> あなたのトレースで見えた値を「あなたの記入」欄に書いてください。

| チェック項目 | 記入例 (参考) | あなたの記入 |
|---|---|---|
| ① ループは何周したか | 0 周 (ツール不要で直接回答) または 1 周 | |
| ② 呼ばれたツール名と引数 | (このターンはツールを呼ばないことが多い) / または get_system_status など | |
| ③ 合計トークン (Total Tokens) | 例: 約 1,200 | |
| 2 ターン目のモデル入力に 1 ターン目の会話 (「人事部の佐藤」) が含まれるか | 含まれている | |
| それは何の働きによるものか | checkpointer が state を復元したため | |

> **読み解きのポイント**: 「さっき伝えた所属はどこ?」はツールを使わずに答えられる質問なので、
> ループ周回は少ない (多くの場合 0 周=ツールなしで直接回答) はずです。一方で、**モデルへの入力**には
> 1 ターン目の「人事部の佐藤です」が含まれています。これが **checkpointer の効果**——
> 「state を復元し、履歴全体がモデルに送られる」——の物的証拠です。
> print では追いにくかったこの事実が、トレースのモデル入力で一目で確認できます。

### (発展) タグでの絞り込みも試す

トレース一覧の検索/フィルタで `tags` に `helpdesk` を指定すると、TODO③ で付けたタグの付いた
トレースだけに絞り込めます。`metadata.department` でのフィルタもできます。
「人事部からの問い合わせだけを追う」といった運用調査が、このタグ・メタデータ設計で可能になります。

---

## まとめ — v1 から v2 へ何が変わったか

| 使った部品 | 学んだ節 | この演習での役割 |
|---|---|---|
| `InMemorySaver` (checkpointer) | 4-2 | `create_agent(..., checkpointer=...)` で会話を記憶 (TODO①) |
| `thread_id` (社員 ID) | 4-2 | 社員ごとに会話を分離して記憶 (TODO②) |
| `tags` / `metadata` | 4-4 | 利用部署をトレースに記録 (TODO③) |
| LangSmith トレース読解 | 4-4 | 2 ターン目の周回数・ツール・トークンを読み、checkpointer の効果を確認 |

### 完成の目安 (達成できたか確認)

- ✅ 佐藤さん (`emp-sato`) の 2 ターンで、2 ターン目に「人事部」と答えられる (記憶の継続 / TODO①②)
- ✅ 別の社員 ID (`emp-tanaka`) では所属を答えられない (記憶の分離)
- ✅ トレースに `tags=["day1","helpdesk"]` と `metadata={"department":"人事部"}` が付く (TODO③)
- ✅ トレースのモデル入力に 1 ターン目の会話が含まれることを確認できた (checkpointer の効果)

> **「なぜ checkpointer なしだと忘れるのか」**——それは LLM API がステートレスで、
> 履歴を保持・再送する層がなかったからです。v1 にはその層がありませんでした。
> checkpointer がその層を担い、`thread_id` を鍵に state を保存・復元します。

### 次章の予告

この v2 は、ツールを**コード内で定義**しています。第5章では、ツールを **MCP** という標準プロトコルで
**外部サーバーから調達**する v3 へ拡張します。会話の記憶 (checkpointer) は、その先の第6章の
**HITL (承認フロー)** でも「中断・再開の基盤」として効いてきます。